In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "/workspaces/dev/modules/python-utils",
    "/workspaces/dev/modules/ai-utils",
    "/workspaces/dev/test/performance_test/esic",
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

In [ ]:
import numpy as np
from pathlib import Path

In [ ]:
from sj_ai_utils.datasets.esic_v1.service import search_dirs
from sj_ai_utils.datasets.esic_v1.sclite import generate_trn
from sj_ai_utils.evaluator.sclite_utils import sclite_trn, parse_sclite_summary
from sj_utils.file.yaml import load_yaml
from sj_utils.collection import SafetyDict
from sj_utils.evaluator import TimeChecker

In [ ]:
from util import get_token_saver_loader_transcriber, normalize_text

In [ ]:
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000
RANDOM_SEED = 42

In [ ]:
SOURCE = "/workspaces/dev/datasets/ESIC-v1.1/v1.1/test"
STORAGE = "/workspaces/dev/storage/esic/"
HYPERPARAMETERS_PATH = "/workspaces/dev/test/performance_test/esic/hyperparameters/20250727/96000/trial_wer4o6_2010_20250727_024423.yaml"

In [ ]:
src = Path(SOURCE)
storage = Path(STORAGE)
hyperparameter_path = Path(HYPERPARAMETERS_PATH)
hyperparameter_path.exists()

In [ ]:
data_paths = search_dirs(src)

In [ ]:
hyperparameter = SafetyDict({
    "whisper": {
        "model_options": {
            "model_size_or_path": "large-v3",
            "device": "cuda",
            "compute_type": "float16",
        },
        "transcribe_options": {
            "beam_size":5,
            "vad_filter": False,
            "temperature": [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
        }
    },
    "silero_vad": {
        "model_options": {},
        "run_options": {}
    },
    "asr": {
        "max_overlap_duration": 16000,
    },
    "position_weighted_filter": {
        "boundary": 0,
    },
    "duration_filter": {
        "z_thresh": {
            "default": 2.0,
            "ko": 2.0,
            "en": 5.0,
        },
        "min_dur": {
            "default": 160,
            "ko": 160,
            "en": 160,
        }
    },
    "probability_filter":{
        "z_thresh":{
            "default": 3.0,
            "ko": 3.0,
            "en": 3.4,
        },
        "min_prob": {
            "default": 1.0,
            "ko": 0.4,
            "en": 0.15
        },
    },
    "selector":{
        "iou_threshold": {
            "default": 0.5,
            "ko": 0.4,
            "en": 0.75,
        },
        "cos_threshold":{
            "default": 0.5,
            "ko": 0.25,
            "en": 0.64
        },
        "padding": {
            "default": 3200,
            "ko": 3200,
            "en": 15800
        },
    },
})

In [ ]:
transcribe_time = TimeChecker()
processed_time = TimeChecker()

In [ ]:
hyperparameter = SafetyDict(load_yaml(hyperparameter_path)[1])
rng = np.random.default_rng(RANDOM_SEED)
_transcriber = get_token_saver_loader_transcriber(
    src,
    storage,
    SAMPLE_RATE,
    rng,
    hyperparameter=hyperparameter,
    overlap = hyperparameter["asr"]["max_overlap_duration"],
)
transcriber = lambda audio: _transcriber(audio, transcribe_time)

In [ ]:
processed_time.start()
data = generate_trn(data_paths, transcriber, normalize_text, 1)
processed_time.check()

In [ ]:
concat_result = {}
for value in data.values():
    for k, v in value.items():
        if k not in concat_result:
            concat_result[k] = []
        concat_result[k].extend(v)

In [ ]:
output = sclite_trn(
    concat_result["ref"],
    concat_result["hyp"],
)

In [ ]:
{
    "result":parse_sclite_summary(output),
    "processed_time": processed_time.metric(),
    "transcribe_time": transcribe_time.metric(),
}